In [ ]:


import re
import pandas as pd
import spacy
import geonamescache
import pycountry
from unidecode import unidecode
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


TEXT_COL = "text"  # <-- change

PLACES_RAW = """
Aalborg
Aarhus
San Bartolomé
İzmir
Ålesund
Akureyri
Málaga
Alghero
Arvidsjaur
Alicante
Alta
Amsterdam
Ancona
Karpathos
Uppsala
Athens
Alexandroupolis
Antalya
Barcelona
Brindisi
Belgrade
Berlin
Bergen
Birmingham
Bastia
Bilbao
Biarritz
Bodrum
Billund
Bologna
Bordeaux
Burgas
Bodø
Bremen
Bari
Bristol
City of Brussels
Mulhouse
Budapest
Bydgoszcz
Cagliari
Paris
Corfu
Bonn
Chania
Cluj-Napoca
Calvi
Copenhagen
Catania
Cavtat
Debrecen
Dalaman
Dresden
Dortmund
Dublin
Düsseldorf
Edinburgh
Cephalonia
Evenes
Yerevan
Faro
Rome
Florence
Osnabrück
Madeira
Frankfurt
Porto-Vecchio
Fuerteventura
Tricity
Graz
Glasgow
Genoa
Gothenburg
Patras
Graz
Geneva
Sylt
Baku
Hanover
Hamburg
Haugesund
Heringsdorf
Helsinki
Heraklion
Iași
Ibiza
Niš
Innsbruck
Innsbruck
Istanbul
Inari
Jersey
Mykonos
Naxos
Skiathos
Santorini
Kuusamo
Keflavík
Kos
Klagenfurt
Kalamata
Kraków
Kiruna
Košice
Kittilä
Katowice
Kavala
Larnaca
London
Leipzig/Halle
London
London
Milan
Lisbon
Ljubljana
Luleå
Linz
Gran Canaria
Luxembourg City
Lemnos
Longyearbyen
Lyon
Linz
Madrid
Mahón
Greater Manchester
Mytilene
Valletta
Montpellier
Marseille
Munich
Lombardy
Naples
Nice
Newcastle upon Tyne
England
Nantes
Nuremberg
Ohrid
Olbia
Oradea
Porto
Cork
Östersund
Eastern Norway
Bucharest
Oulu
Oviedo
Lippstadt
South Aegean
Ponta Delgada
Paphos
Mallorca
Palermo
Poznań
Prague
Pristina
Pisa
Pula
Preveza
Düsseldorf
Altstadt-Nord
Germany
Reggio Calabria
Rhodes
Riga
Rijeka
Chișinău
Rovaniemi
Rzeszów
Sibiu
Galicia
Skellefteå
Istočno Sarajevo
Thessaloniki
Skopje
Vathy
Limerick
Stolichna
Santa Cruz de La Palma
Split
London
Stuttgart
Lamezia Terme
Stavanger
Seville
Salzburg
Tbilisi
Praia da Vitória
Tenerife
Podgorica
Tirana
Tivat
Tallinn
Toulouse
Tromsø
Stjørdal
Turin
Trieste
Timiș County
Umeå
Varna
Venice
Vienna
Valencia
Vilnius
Verona
Warsaw
Wrocław
Strasbourg
Jerez de la Frontera
Zadar
Zagreb
Nuremberg
Basel
Bern
Geneva
Munich
Zurich
Salzburg
Zakynthos
Stuttgart-Center
""".strip()

# ------------------ helpers ------------------

def norm(s: str) -> str:
    s = (s or "").strip()
    s = unidecode(s)                     # remove diacritics
    s = s.lower()
    s = re.sub(r"[’'`]", "", s)
    s = re.sub(r"[^a-z0-9/ -]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def canon_country(name_or_code: str | None) -> str | None:
    if not name_or_code:
        return None
    x = str(name_or_code).strip().lower()
    return country_map.get(x)


def country_iso(cc: str):
    try:
        c = pycountry.countries.lookup(cc)
        return c.alpha_2
    except Exception:
        return None

# ------------------ build gazetteer ------------------

# canonical place list (dedup, keep original casing)
places = []
seen = set()
for line in PLACES_RAW.splitlines():
    p = line.strip()
    if not p:
        continue
    k = norm(p)
    if k not in seen:
        places.append(p)
        seen.add(k)

gc = geonamescache.GeonamesCache()
cities = gc.get_cities()
countries = gc.get_countries()  # iso2 -> {name:...}

iso2_to_countryname = {k: v["name"] for k, v in countries.items()}

# country canonicalization map
country_map = {}
for c in pycountry.countries:
    keys = [c.name, getattr(c, "official_name", None), getattr(c, "common_name", None),
            getattr(c, "alpha_2", None), getattr(c, "alpha_3", None)]
    for key in filter(None, keys):
        country_map[str(key).strip().lower()] = c.name


# manual fixes + “entities that are not cities but you want tracked”
# (you can edit these freely)
OVERRIDES = {
    "city of brussels": {"canonical": "City of Brussels", "country": "Belgium", "iso2": "BE", "type": "city",
                         "aliases": ["City of Brussels", "Brussels", "Bruxelles", "Brussel"]},
    "greater manchester": {"canonical": "Greater Manchester", "country": "United Kingdom", "iso2": "GB", "type": "metro",
                           "aliases": ["Greater Manchester", "Manchester"]},
    "tricity": {"canonical": "Tricity", "country": "Poland", "iso2": "PL", "type": "metro",
                "aliases": ["Tricity", "Tri-City", "Trojmiasto", "Trójmiasto", "Gdansk", "Gdańsk", "Gdynia", "Sopot"]},
    "leipzig/halle": {"canonical": "Leipzig/Halle", "country": "Germany", "iso2": "DE", "type": "airport_area",
                      "aliases": ["Leipzig/Halle", "Leipzig Halle", "Leipzig–Halle", "Leipzig", "Halle"]},
    "lippstadt": {"canonical": "Lippstadt", "country": "Germany", "iso2": "DE", "type": "city",
                  "aliases": ["Lippstadt", "Paderborn", "Paderborn/Lippstadt", "Paderborn Lippstadt"]},
    "stuttgart-center": {"canonical": "Stuttgart-Center", "country": "Germany", "iso2": "DE", "type": "district",
                         "aliases": ["Stuttgart-Center", "Stuttgart Center", "Stuttgart-Mitte", "Stuttgart Mitte"]},
    "altstadt-nord": {"canonical": "Altstadt-Nord", "country": "Germany", "iso2": "DE", "type": "district",
                      "aliases": ["Altstadt-Nord", "Altstadt Nord"]},
    "eastern norway": {"canonical": "Eastern Norway", "country": "Norway", "iso2": "NO", "type": "region",
                       "aliases": ["Eastern Norway", "Østlandet", "Ostlandet"]},
    "south aegean": {"canonical": "South Aegean", "country": "Greece", "iso2": "GR", "type": "region",
                     "aliases": ["South Aegean", "Notio Aigaio", "Notío Aigaío", "Notio Aigaío"]},
    "lombardy": {"canonical": "Lombardy", "country": "Italy", "iso2": "IT", "type": "region",
                 "aliases": ["Lombardy", "Lombardia"]},
    "galicia": {"canonical": "Galicia", "country": "Spain", "iso2": "ES", "type": "region",
                "aliases": ["Galicia"]},
    "timis county": {"canonical": "Timiș County", "country": "Romania", "iso2": "RO", "type": "county",
                     "aliases": ["Timiș County", "Timis County", "Timiș", "Timis"]},
    "madeira": {"canonical": "Madeira", "country": "Portugal", "iso2": "PT", "type": "region",
                "aliases": ["Madeira", "Funchal"]},
    "england": {"canonical": "England", "country": "United Kingdom", "iso2": "GB", "type": "country_part",
                "aliases": ["England"]},
    "germany": {"canonical": "Germany", "country": "Germany", "iso2": "DE", "type": "country",
                "aliases": ["Germany", "Deutschland"]},
    "jersey": {"canonical": "Jersey", "country": "Jersey", "iso2": "JE", "type": "country_part",
               "aliases": ["Jersey", "Channel Islands"]},
    # ambiguous/rare token in your list; keep track but don’t force a country guess
    "stolichna": {"canonical": "Stolichna", "country": None, "iso2": None, "type": "unknown",
                  "aliases": ["Stolichna"]},
    "france": {"canonical": "France", "country": "France", "iso2": "FR", "type": "country",
                "aliases": ["Francia", "France", "French"]},
    "europe": {"canonical":"Europe","country":None,"iso2":None,"type":"continent","aliases":["Europe","EU","European Union"]},
    "switzerland": {"canonical":"Switzerland","country":"Switzerland","iso2":"CH","type":"country",
                    "aliases":["Switzerland","Schweiz","Suisse","Svizzera"]},
    "netherlands": {"canonical":"Netherlands","country":"Netherlands","iso2":"NL","type":"country",
                    "aliases":["Netherlands","The Netherlands","Holland", "Dutch"]},
    "portugal": {"canonical":"Portugal","country":"Portugal","iso2":"PT","type":"country",
                 "aliases":["Portugal"]},
    "spain": {"canonical":"Spain","country":"Spain","iso2":"ES","type":"country",
              "aliases":["Spain","España","Espana"]},
}

# Build geonames cache index: city name -> candidates
city_index = {}
for c in cities.values():
    nm = (c.get("name") or "").strip()
    if nm:
        city_index.setdefault(norm(nm), []).append(c)

# Extra alias rules (common English/local name swaps + diacritics forms)
EXTRA_ALIASES = {
    "dusseldorf": ["Düsseldorf", "Dusseldorf"],
    "osnabruck": ["Osnabrück", "Osnabruck"],
    "krakow": ["Kraków", "Krakow"],
    "poznan": ["Poznań", "Poznan"],
    "wroclaw": ["Wrocław", "Wroclaw"],
    "chisinau": ["Chișinău", "Chisinau"],
    "iasi": ["Iași", "Iasi"],
    "nis": ["Niš", "Nis"],
    "tromso": ["Tromsø", "Tromso"],
    "bodo": ["Bodø", "Bodo"],
    "lulea": ["Luleå", "Lulea"],
    "ostersund": ["Östersund", "Ostersund"],
    "umea": ["Umeå", "Umea"],
    "kittila": ["Kittilä", "Kittila"],
    "keflavik": ["Keflavík", "Keflavik"],
    "izmir": ["İzmir", "Izmir"],
    "malaga": ["Málaga", "Malaga"],
    "mahon": ["Mahón", "Mahon", "Maó", "Mao"],
    "geneva": ["Geneva", "Genève", "Geneve"],
    "basel": ["Basel", "Bâle", "Bale"],
    "vienna": ["Vienna", "Wien"],
    "florence": ["Florence", "Firenze"],
    "rome": ["Rome", "Roma"],
    "milan": ["Milan", "Milano"],
    "naples": ["Naples", "Napoli"],
    "venice": ["Venice", "Venezia"],
    "turin": ["Turin", "Torino"],
    "genoa": ["Genoa", "Genova"],
    "seville": ["Seville", "Sevilla"],
    "lisbon": ["Lisbon", "Lisboa"],
    "porto": ["Porto", "Oporto"],
    "gothenburg": ["Gothenburg", "Göteborg", "Goteborg"],
    "copenhagen": ["Copenhagen", "København", "Kobenhavn"],
    "helsinki": ["Helsinki", "Helsingfors"],
    "ibiza": ["Ibiza", "Eivissa"],
    "mallorca": ["Mallorca", "Majorca", "Palma", "Palma de Mallorca"],
    "corfu": ["Corfu", "Kerkyra"],
    "rhodes": ["Rhodes", "Rhódos", "Rhodos", "Rodos"],
    "santorini": ["Santorini", "Thira", "Thera"],
    "zurich": ["ZÃ¼rich"]
}

# Final gazetteer dict keyed by canonical
# Each value: {"country":..,"iso2":..,"type":..,"aliases":[...]}
GAZETTEER = {}

def choose_best_candidate(cands):
    # prefer Europe-ish countries by simple whitelist of ISO2 seen commonly in your list
    preferred_iso2 = {"DK","NO","SE","FI","IS","NL","BE","LU","GB","IE","FR","DE","CH","AT","ES","PT","IT","GR","TR","CY",
                      "PL","CZ","SK","HU","RO","BG","HR","SI","RS","BA","ME","AL","MK","LV","LT","EE","MD","MT","GE","AM","AZ"}
    for c in cands:
        if c.get("countrycode") in preferred_iso2:
            return c
    return cands[0] if cands else None

for place in places:
    k = norm(place)

    if k in OVERRIDES:
        rec = OVERRIDES[k].copy()
        GAZETTEER[rec["canonical"]] = {
            "country": rec.get("country"),
            "iso2": rec.get("iso2"),
            "type": rec.get("type", "city"),
            "aliases": sorted(set(rec.get("aliases", []) + [rec["canonical"]]))
        }
        continue

    cands = city_index.get(k, [])
    best = choose_best_candidate(cands)

    if best:
        iso2 = best.get("countrycode")
        country = canon_country(iso2_to_countryname.get(iso2)) or iso2_to_countryname.get(iso2)
        canonical = best.get("name") or place
        GAZETTEER[canonical] = {
            "country": country,
            "iso2": iso2,
            "type": "city",
            "aliases": [canonical, place]
        }
    else:
        # fallback: keep as trackable, unknown country
        GAZETTEER[place] = {"country": None, "iso2": None, "type": "unknown", "aliases": [place]}

# Add extra aliases into the correct canonical buckets
# (If a canonical doesn’t exist yet, it will be created.)
for _, alias_list in EXTRA_ALIASES.items():
    canonical = alias_list[0]
    if canonical not in GAZETTEER:
        GAZETTEER[canonical] = {"country": None, "iso2": None, "type": "unknown", "aliases": []}
    GAZETTEER[canonical]["aliases"] = sorted(set(GAZETTEER[canonical]["aliases"] + alias_list))

# Build alias -> canonical for matching
alias_to_canonical = {}
for canonical, rec in GAZETTEER.items():
    for a in rec.get("aliases", []) + [canonical]:
        alias_to_canonical[norm(a)] = canonical

# ------------------ spaCy: EntityRuler using gazetteer aliases ------------------

nlp = spacy.load("en_core_web_sm", disable=["parser","tagger","lemmatizer"])
ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = [{"label": "GPE", "pattern": a} for a in sorted({a for rec in GAZETTEER.values() for a in rec["aliases"]})]
ruler.add_patterns(patterns)

sent = SentimentIntensityAnalyzer()

def sent_label(comp):
    return "positive" if comp >= 0.05 else ("negative" if comp <= -0.05 else "neutral")

def extract_places_and_sentiment(text: str, max_hits=30):
    if not isinstance(text, str) or not text.strip():
        return [], "neutral", 0.0
    # text = text or ""
    doc = nlp(text)

    hits = []
    seen = set()
    for ent in doc.ents:
        if ent.label_ in ("GPE","LOC"):
            raw = ent.text.strip()
            key = norm(raw)
            if not key or key in seen:
                continue
            seen.add(key)

            cc = canon_country(raw)
            if cc:
                hits.append({"mention": raw, "canonical": cc, "country": cc, "iso2": country_iso(cc), "type":"country"})
                continue

            canonical = alias_to_canonical.get(key)
            if canonical:
                rec = GAZETTEER[canonical]
                hits.append({
                    "mention": raw,
                    "canonical": canonical,
                    "country": rec["country"],
                    "iso2": rec["iso2"],
                    "type": rec["type"],
                })

    comp = float(sent.polarity_scores(text)["compound"])
    return hits[:max_hits], sent_label(comp), comp

# ------------------ Apply to dataframe ------------------

# df = pd.read_excel("uktravel_processed.xlsx")
df = pd.read_csv("test.csv")
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
out = df[TEXT_COL].apply(lambda t: pd.Series(extract_places_and_sentiment(t),
                                            index=["locations","sentiment_label","sentiment_compound"]))
df = df.join(out)

# Optional: explode locations into a long table
loc_df = (df.assign(row_id=df.index)
            .explode("locations")
            .dropna(subset=["locations"]))
loc_df = pd.concat([loc_df[["row_id"]], loc_df["locations"].apply(pd.Series)], axis=1)

print("GAZETTEER size:", len(GAZETTEER), "canonical entries")
print(df[[TEXT_COL,"sentiment_label","sentiment_compound","locations"]].head(3))
print(loc_df.head(10))

GAZETTEER size: 214 canonical entries
                                                text sentiment_label  \
0  MoMA vs the Ride NYC Hi guys, I’m planning to ...        positive   
1  Stolen Belongings from Benito Juarez (MEX) I l...        negative   
2  Good out of U.S. locations for chill bachelor ...        positive   

   sentiment_compound locations  
0              0.8070        []  
1             -0.7644        []  
2              0.4404        []  
    row_id      mention           canonical             country  iso2     type
6        6       London              London      United Kingdom    GB     city
8        8           US       United States       United States    US  country
10      10       Canada              Canada              Canada    CA  country
10      10           US       United States       United States    US  country
11      11        Turin               Turin               Italy    IT     city
11      11        Malta               Malta               Malta

In [2]:
merged_df = loc_df.merge(
    df.drop(columns=["locations", "city", "country"], errors="ignore").reset_index().rename(columns={"index": "row_id"}),
    on="row_id",
    how="left"
)

In [3]:
import re

illegal_chars = re.compile(r'[\000-\010]|[\013-\014]|[\016-\037]')

def clean_excel_text(x):
    if isinstance(x, str):
        return illegal_chars.sub("", x)
    return x

merged_df = merged_df.applymap(clean_excel_text)

C:\Users\myq14\AppData\Local\Temp\ipykernel_49532\651384951.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  merged_df = merged_df.applymap(clean_excel_text)


In [ ]:
merged_df.to_excel("testwith_locations.xlsx")